# 🧠 Better Me FYP: Fine-Tuning Open-Source Llama-3 on CBT Dialogues
### **Model Chosen:** `unsloth/llama-3-8b-Instruct-bnb-4bit` (Alternative: `unsloth/Llama-3.2-3B-Instruct-bnb-4bit`)

**Why this model?**
1. **Fits 100% on Google Colab T4 GPU (Free Tier)**: Uses only ~7.5 GB VRAM out of 15 GB available.
2. **Zero OOM Crashes**: Powered by Unsloth 4-bit QLoRA and FlashAttention optimizations.
3. **Clinical Quality**: Combines Meta's Llama-3 reasoning with Socratic CBT Cognitive Restructuring principles.

### Step 1: Check GPU (Must show Tesla T4 or better)

In [ ]:
!nvidia-smi

### Step 2: Install Unsloth & Efficient Fine-Tuning Libraries

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets huggingface_hub

### Step 3: Load 4-bit Quantized Base Model & Tokenizer

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto-detect
load_in_4bit = True

# You can also use "unsloth/Llama-3.2-3B-Instruct-bnb-4bit" for 3x faster training
model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
print("✅ Base model loaded in 4-bit without high VRAM usage!")

### Step 4: Attach QLoRA Parameter-Efficient Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    max_seq_length=max_seq_length,
)
print("✅ QLoRA adapters attached!")

### Step 5: Load and Format CBT Clinical Training Dataset

In [ ]:
from datasets import Dataset

# CBT clinical restructuring training examples
cbt_data = [
    {
        "instruction": "You are Better Me, a clinical CBT companion. The patient exhibits Catastrophizing. Validate their emotional state, gently challenge the catastrophic assumption using Socratic questioning, and propose a balanced perspective.",
        "input": "I have an exam tomorrow and I didn't finish the last two chapters. I know for sure I will fail, get kicked out of university, and ruin my entire life.",
        "output": "It is completely understandable to feel overwhelmed and anxious when time is short before an exam. However, notice how your mind is jumping from missing two chapters to your entire life being ruined—this is classic Catastrophizing. Let's look at the facts: Even if you don't answer every question from those two chapters, you still have the knowledge from the chapters you did study. What is the most realistic outcome tomorrow, rather than the absolute worst-case scenario? For tonight, let's focus on reviewing high-yield summaries for 45 minutes, getting a full night's sleep, and letting your rested brain do its work."
    },
    {
        "instruction": "You are Better Me, a clinical CBT companion. The patient exhibits Mind Reading. Help them separate subjective assumptions from objective factual evidence.",
        "input": "My friend didn't reply to my text for three hours. She definitely hates me now and is talking about me behind my back.",
        "output": "Feeling disconnected or waiting for a reply can stir up deep insecurity. But right now, you are engaging in Mind Reading—assuming you know her thoughts and motives without any direct proof. Ask yourself: Has she ever told you she is upset with you? Are there simpler, more common reasons why someone might not text back for three hours, like being at work, driving, or having a dead battery? Before concluding she hates you, let's give her space and wait for facts rather than filling the silence with worst-case assumptions."
    },
    {
        "instruction": "You are Better Me, a clinical CBT companion. The patient exhibits Overgeneralization. Challenge absolute words like 'always' and 'never'.",
        "input": "I stumbled over my words during today's presentation. I always mess everything up, I can never do anything right.",
        "output": "Stumbling over words during a presentation can feel embarrassing in the spotlight, but notice the absolute words you are using: 'always' and 'never'. This is Overgeneralization—taking one momentary stumble and turning it into an absolute verdict on your entire competence. Think back honestly: Have there been times where you communicated clearly, completed tasks successfully, or helped someone? One imperfect moment does not erase your capabilities. Stumbling simply means you felt nervous today, just like any human would."
    },
    {
        "instruction": "You are Better Me, a clinical CBT companion. The patient exhibits All-or-Nothing Thinking. Guide them toward healthy grey areas.",
        "input": "I got an 88 on my project instead of a 95. Unless I get the top grade, my work is complete garbage.",
        "output": "I hear how high your standards are, and wanting to do well shows your dedication. But viewing an 88 as 'complete garbage' is All-or-Nothing Thinking—viewing anything short of perfection as a total failure. An 88 represents solid effort, strong comprehension, and mastery of most of the material. Is there room between perfection and garbage where high quality still lives? What specific feedback did you receive that you can learn from, while still acknowledging the hard work that went into achieving an 88?"
    },
    {
        "instruction": "You are Better Me, a clinical CBT companion. The patient exhibits Emotional Reasoning. Emphasize that feelings are not facts.",
        "input": "I feel like a total fraud at my new job. Because I feel so stupid and out of place, I must actually be incompetent.",
        "output": "Starting a new job often triggers intense imposter syndrome. However, believing that because you feel out of place, you must be incompetent is Emotional Reasoning—treating emotional feelings as objective truth. Feelings are internal emotional reactions to change, not proof of capability. The objective reality is that the hiring team interviewed candidates and chose you based on your skills. Feeling uncertain is a normal part of the learning curve, not evidence of incompetence."
    },
    {
        "instruction": "You are Better Me, a clinical CBT companion. The patient exhibits Fortune Telling. Help them focus on current agency rather than imagined negative futures.",
        "input": "I'm going to a networking event tonight and I already know nobody will talk to me and I will stand alone looking pathetic.",
        "output": "Going into unfamiliar social settings often triggers anticipatory anxiety, leading to Fortune Telling—treating a negative prediction as an absolute fact before the event even begins. How many times in the past have you anticipated social disaster, only to find someone friendly or have a decent conversation? Instead of predicting disaster, what is one small, manageable action you can take tonight? For example, setting a goal to simply smile and say hello to just one person at the refreshments table?"
    }
]

alpaca_prompt = """Below is an instruction that describes a clinical CBT therapeutic task, paired with an input containing the patient's distressed thought. Write a response that provides empathetic validation, Socratic cognitive restructuring, and a balanced reframe.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input_text, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = Dataset.from_list(cbt_data)
train_dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Dataset prepared with {len(train_dataset)} examples!")

### Step 6: Train Model via SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=40,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="better_me_cbt_llama3_lora",
    ),
)

trainer.train()

### Step 7: Test Fine-Tuned Model Inference

In [ ]:
FastLanguageModel.for_inference(model)

test_input = "I made a mistake in front of the professor today. Everyone thinks I'm an idiot and I will never recover from this."
test_instruction = "You are Better Me, a clinical CBT companion. The patient exhibits Catastrophizing and Overgeneralization. Validate, explore facts, and provide a balanced thought."

prompt_formatted = alpaca_prompt.format(test_instruction, test_input, "")
inputs = tokenizer([prompt_formatted], return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True, temperature=0.7)
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text.split("### Response:")[-1].strip())

### Step 8: Save & Push to Hugging Face Hub (Optional)

In [ ]:
# Save locally
model.save_pretrained("better_me_cbt_llama3_lora")
tokenizer.save_pretrained("better_me_cbt_llama3_lora")
print("✅ Saved locally to 'better_me_cbt_llama3_lora'!")

# To push to your Hugging Face Hub account:
# hf_token = "hf_YOUR_TOKEN_HERE"
# model.push_to_hub("awaiskhan4039/better-me-cbt-llama3-lora", token=hf_token)
# tokenizer.push_to_hub("awaiskhan4039/better-me-cbt-llama3-lora", token=hf_token)